In [10]:
import pandas as pd
import numpy as np
import os

def backtest(average_scores_df, stock):
    average_scores_df = average_scores_df[['datetime', 'instrument', 'score']]
    average_scores_df['datetime'] = average_scores_df['datetime'].astype('datetime64[ns]')
    average_scores_df = average_scores_df.sort_values(by=['datetime', 'instrument'], ascending=[True, True])
    average_scores_df = average_scores_df[average_scores_df['instrument'].isin(stock)]

    stock_df = pd.read_csv('/home/liyuante/cikm2024/dataset/zz500_2018_2023_new_2.csv')
    stock_df['dt'] = pd.to_datetime(stock_df['dt'])
    stock_df = stock_df[['kdcode', 'dt', 'close']]
    stock_df = stock_df.rename(columns={'kdcode': 'instrument', 'dt': 'datetime'})
    stock_df = stock_df.sort_values(by=['instrument', 'datetime'])
    stock_df['close_r'] = stock_df['close'] / stock_df['close'].shift(1)
    df_trade = stock_df[stock_df['datetime'] > '2022-12-31']

    x = pd.merge(df_trade, average_scores_df, on=['datetime', 'instrument'], how='outer')
    trade_date_unique = df_trade['datetime'].unique()
    df_return = pd.DataFrame()

    for date in trade_date_unique:
        x_day = x[x['datetime'] == date]
        r_day = x_day.nlargest(10, columns='score').close_r.mean()
        r_day = r_day - 1
        b = {'datetime': date, 'daily_return': r_day}
        df_return = df_return.append(b, ignore_index=True)

    portfolio_df_performance = df_return.set_index(['datetime'])

    alpha_df_performance = pd.DataFrame()
    alpha_df_performance['portfolio_daily_return'] = portfolio_df_performance['daily_return']
    alpha_df_performance['portfolio_net_value'] = (alpha_df_performance['portfolio_daily_return'] + 1).cumprod()

    net_value_columns = ['portfolio_net_value']

    alpha_statistics_df = pd.DataFrame(index=alpha_df_performance[net_value_columns].columns,
                                        columns=["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"])

    alpha_df_performance.index = pd.to_datetime(alpha_df_performance.index)
    alpha_statistics_df.loc[:, "年化收益"] = np.mean((alpha_df_performance[net_value_columns].tail(1)) ** (252 / len(alpha_df_performance)) - 1)
    alpha_statistics_df.loc[:, "年化波动率"] = np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252)
    alpha_statistics_df.loc[:, "最大回撤率"] = np.min((alpha_df_performance[net_value_columns] - alpha_df_performance[net_value_columns].cummax()) / alpha_df_performance[net_value_columns].cummax())
    alpha_statistics_df.loc[:, "夏普率"] = alpha_statistics_df["年化收益"] / alpha_statistics_df["年化波动率"]
    alpha_statistics_df.loc[:, "Calmar"] = alpha_statistics_df["年化收益"] / abs(alpha_statistics_df["最大回撤率"])
    alpha_statistics_df.loc[:, "IR"] = np.mean(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252) / np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1)

    return alpha_statistics_df

def get_merged_score_df(base_path, prediction_index):
    score_dfs = []
    prediction_path = os.path.join(base_path, f'prediction_{prediction_index}')
    
    for j in range(5):
        folder_path = os.path.join(prediction_path, str(j))
        all_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]
        
        for file in all_files:
            df = pd.read_csv(file)
            score_dfs.append(df)
    
    # 合并所有score_dfs并取均值
    merged_df = pd.concat(score_dfs).groupby(['dt', 'kdcode']).mean().reset_index()
    merged_df.rename(columns={'kdcode': 'instrument', 'dt' : 'datetime'}, inplace=True)
    return merged_df

# 主程序
base_path = '/home/liyuante/2_20240707_csi500_0.7_5_10'
d = pd.read_csv('/home/liyuante/cikm2024/dataset/zz500_2018_2023_new_2.csv')
stock = d['kdcode'].unique()

results = []

for i in range(20):
    average_scores_df = get_merged_score_df(base_path, i)
    result = backtest(average_scores_df, stock)
    result = result[["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"]]  # 只保留6个指标
    result['folder'] = i
    results.append(result)

# 显示结果
final_df = pd.concat(results).reset_index(drop=True)
print(final_df)


        年化收益     年化波动率     最大回撤率       夏普率    Calmar        IR  folder
0  -0.060401  0.156661 -0.179077 -0.385553 -0.337292 -0.371325       0
1  -0.095963  0.159751 -0.219505 -0.600707 -0.437181 -0.612719       1
2  -0.116076  0.185578 -0.326381 -0.625481 -0.355645 -0.625411       2
3  -0.098549  0.173286 -0.266011 -0.568707 -0.370471 -0.539362       3
4  -0.065187  0.177240 -0.255003 -0.367787 -0.255630 -0.322718       4
5  -0.266188  0.182410 -0.329418 -1.459288 -0.808056 -1.666378       5
6  -0.009962  0.188967 -0.226657 -0.052717 -0.043951 -0.026016       6
7   0.211534  0.166939 -0.169897  1.267134  1.245073  1.216992       7
8   0.291057  0.175385 -0.108684  1.659527  2.678011  1.515183       8
9  -0.016001  0.156437 -0.276609 -0.102284 -0.057847 -0.053285       9
10 -0.152948  0.169953 -0.275202 -0.899944 -0.555765 -0.920656      10
11  0.056006  0.167822 -0.202348  0.333722  0.276780  0.355624      11
12  0.225410  0.180399 -0.154897  1.249509  1.455224  1.103374      12
13  0.